In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Specify the column names to ensure consistency
column_names = ['ep (ms)', 'Acc_x', 'Acc_y', 'Acc_z', 'Gyro_x', 'Gyro_y', 'Gyro_z', 'ID', 'Label', 'Category', 'Set']

# Read the CSV while specifying column names
df = pd.read_csv('data2.csv', names=column_names, skiprows=1,sep=',')


In [6]:
df
num_rows = df.shape[0]
num_rows
k = 1+(10/3) * np.log10(num_rows)
k


np.float64(14.182255289728813)

In [19]:
def descretization(df) : 
    k = 1+(10/3) * np.log10(num_rows)
    df_bins = pd.DataFrame()
    df_intervals = pd.DataFrame()

    for column in df.columns:
        if column != 'ID' and column != 'Label' and column != 'Category' and column != 'Set' and column != 'ep (ms)':
            MaxValue = df[column].max()
            MinValue = df[column].min()
            width = (MaxValue - MinValue) / k
            bins=np.arange(MinValue, MaxValue, width)
            df_bins[column] = pd.cut(df[column], bins=bins, labels=[f'Bin_{i+1}' for i in range(len(bins) - 1)])
            df_intervals[column] = pd.cut(df[column], bins=bins).astype(str)

    return df_bins , df_intervals


In [20]:
bins ,intervals = descretization(df)


In [21]:
intervals.to_csv('intervals.csv', index=False)
bins.to_csv('bins.csv', index=False)

In [36]:
def calculate_average_from_intervals(df):
    # Initialize a new DataFrame to store the average values
    df_avg = pd.DataFrame()

    # Iterate over each column in the DataFrame
    for column in df.columns:
        # Calculate the average for each interval in the column
        df_avg[column] = df[column].apply(lambda x: calculate_midpoint(x))

    return df_avg

def calculate_midpoint(interval):
    try:
        # Ensure the interval is a string and in the correct format
        if isinstance(interval, str) and ',' in interval and len(interval.split(',')) == 2:
            lower_bound = float(interval.split(',')[0].strip('() '))  # Remove '(' and convert to float
            upper_bound = float(interval.split(',')[1].strip('] '))  # Remove ']' and convert to float
            # Calculate and return the midpoint
            return (lower_bound + upper_bound) / 2
        else:
            # Handle cases where the interval is invalid
            return None  # or return a default value like 0, depending on your needs
    except (ValueError, IndexError) as e:
        print(f"Error processing interval '{interval}': {e}")
        return None  # or a default value

In [37]:
df_avg = calculate_average_from_intervals(intervals)
df_avg.to_csv('average.csv', index=False)

In [44]:
def normalize_custom_range(df, new_min=0, new_max=1):
    df_normalized = pd.DataFrame()

    for column in df.columns:
        if column not in ['ep (ms)' ,'ID', 'Label', 'Category', 'Set']:
            value_min_old = df[column].min()
            value_max_old = df[column].max()
            
            
            df_normalized[column] = (
                (df[column] - value_min_old) / (value_max_old - value_min_old) *
                (new_max - new_min) + new_min
            )

    return df_normalized

In [45]:
df_normalized = normalize_custom_range(df)
df_normalized.to_csv('normalized.csv', index=False)